In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

In [2]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori,association_rules


In [3]:
dataset = [
    ['Bread', 'Milk'],
    ['Bread', 'Diaper', 'Beer', 'Eggs'],
    ['Milk', 'Diaper', 'Beer', 'Cola'],
    ['Bread', 'Milk', 'Diaper', 'Beer'],
    ['Milk', 'Milk', 'Diaper', 'Cola'],
]



In [5]:
for i, t in enumerate(dataset, 1):
    print(f"T{i}: {t}")


#one hot encoding
te = TransactionEncoder()
onehot = te.fit(dataset).transform(dataset)
df = pd.DataFrame(onehot, columns=te.columns_)
df.columns = [str(c) for c in df.columns]
df
print()
print(df)
print()
print("Shape (rows, columns):", df.shape)

T1: ['Bread', 'Milk']
T2: ['Bread', 'Diaper', 'Beer', 'Eggs']
T3: ['Milk', 'Diaper', 'Beer', 'Cola']
T4: ['Bread', 'Milk', 'Diaper', 'Beer']
T5: ['Milk', 'Milk', 'Diaper', 'Cola']

    Beer  Bread   Cola  Diaper   Eggs   Milk
0  False   True  False   False  False   True
1   True   True  False    True   True  False
2   True  False   True    True  False   True
3   True   True  False    True  False   True
4  False  False   True    True  False   True

Shape (rows, columns): (5, 6)


In [6]:
#lesson 2: support
n = len(df)
milk_diaper = df['Milk'] & df['Diaper']

print()
print("Milk & diaper column:")
print(milk_diaper)
print("count(Milk & Diaper) =", milk_diaper.sum())
print("support(Milk, Diaper) =", milk_diaper.sum()/n)

print()
print("Support of each single item:")
print(df.mean())


Milk & diaper column:
0    False
1    False
2     True
3     True
4     True
dtype: bool
count(Milk & Diaper) = 3
support(Milk, Diaper) = 0.6

Support of each single item:
Beer      0.6
Bread     0.6
Cola      0.4
Diaper    0.8
Eggs      0.2
Milk      0.8
dtype: float64


In [7]:
frequent = apriori(df, min_support = 0.4, use_colnames = True)


In [8]:
frequent['length'] = frequent['itemsets'].apply(len)

In [9]:
frequent = frequent.sort_values(['length', 'support'], ascending = [True, False]).reset_index(drop=True)

In [10]:
frequent['length'].value_counts().sort_index()

length
1    5
2    8
3    3
Name: count, dtype: int64

In [11]:
print(f"Total frequent itemsets: {len(frequent)}")
print(frequent)
print()
print("Number of frequent itemsets by length:")


Total frequent itemsets: 16
    support                          itemsets  length
0       0.8               frozenset({Diaper})       1
1       0.8                 frozenset({Milk})       1
2       0.6                 frozenset({Beer})       1
3       0.6                frozenset({Bread})       1
4       0.4                 frozenset({Cola})       1
5       0.6         frozenset({Diaper, Beer})       2
6       0.6         frozenset({Diaper, Milk})       2
7       0.4          frozenset({Bread, Beer})       2
8       0.4           frozenset({Milk, Beer})       2
9       0.4        frozenset({Bread, Diaper})       2
10      0.4          frozenset({Bread, Milk})       2
11      0.4         frozenset({Cola, Diaper})       2
12      0.4           frozenset({Cola, Milk})       2
13      0.4  frozenset({Bread, Diaper, Beer})       3
14      0.4   frozenset({Diaper, Milk, Beer})       3
15      0.4   frozenset({Cola, Milk, Diaper})       3

Number of frequent itemsets by length:


In [12]:
rules = association_rules(frequent, metric='confidence', min_threshold=0.7)

show_cols = ['antecedents', 'consequents','support', 'confidence', 'lift','leverage','conviction']

rules_view = rules[show_cols].sort_values('confidence',ascending = False).reset_index(drop= True)
print()
print(f"Strong rules (conf >= 0.7): {len(rules)}")
print(rules_view)

# Look closely at one rule: {Beer, Milk} => {Diaper}
mask = rules['antecedents'].apply(lambda s: set(s) == {'Beer', 'Milk'}) & \
       rules['consequents'].apply(lambda s: set(s) == {'Diaper'})

print()
print("Rule {Beer, Milk} => {Diaper}:")
print(rules.loc[mask, show_cols])


Strong rules (conf >= 0.7): 12
                   antecedents                consequents  support  \
0            frozenset({Beer})        frozenset({Diaper})      0.6   
1   frozenset({Bread, Diaper})          frozenset({Beer})      0.4   
2            frozenset({Cola})          frozenset({Milk})      0.4   
3            frozenset({Cola})        frozenset({Diaper})      0.4   
4      frozenset({Milk, Beer})        frozenset({Diaper})      0.4   
5      frozenset({Cola, Milk})        frozenset({Diaper})      0.4   
6    frozenset({Cola, Diaper})          frozenset({Milk})      0.4   
7     frozenset({Bread, Beer})        frozenset({Diaper})      0.4   
8            frozenset({Cola})  frozenset({Diaper, Milk})      0.4   
9            frozenset({Milk})        frozenset({Diaper})      0.6   
10         frozenset({Diaper})          frozenset({Beer})      0.6   
11         frozenset({Diaper})          frozenset({Milk})      0.6   

    confidence      lift  leverage  conviction  
0       